In [ ]:
# Install once if needed:
# %pip install kagglehub pandas numpy matplotlib scikit-learn imbalanced-learn xgboost

import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [ ]:
dataset_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
print('Dataset folder:', dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, '**', '*.csv'), recursive=True)
if not csv_files:
    raise FileNotFoundError('No CSV file found.')

csv_path = next((p for p in csv_files if os.path.basename(p).lower() == 'creditcard.csv'), csv_files[0])
df = pd.read_csv(csv_path)

print('Loaded file:', csv_path)
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
print(df['Class'].value_counts())
print('\nPercentages:')
print((df['Class'].value_counts(normalize=True) * 100).round(4))

df['Class'].value_counts().plot(kind='bar', title='Class Distribution')
plt.xlabel('Class: 0 = Legitimate, 1 = Fraud')
plt.ylabel('Number of transactions')
plt.show()


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)


In [ ]:
print('Before SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_resampled).value_counts())


In [ ]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train_resampled, y_train_resampled)
print('XGBoost training completed.')


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
results = pd.DataFrame({'actual_class': y_test.values, 'fraud_probability': y_prob})
display(results.head(10))


In [ ]:
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'PR-AUC:  {average_precision_score(y_test, y_prob):.4f}')


In [ ]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)
    threshold_results.append({
        'threshold': threshold,
        'precision': precision_score(y_test, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_test, y_pred_threshold, zero_division=0),
        'f1_score': f1_score(y_test, y_pred_threshold, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


In [ ]:
chosen_threshold = 0.30
y_pred = (y_prob >= chosen_threshold).astype(int)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.xlabel('Feature importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()
